<a href="https://colab.research.google.com/github/shravan1808/ML_SERIES/blob/Main/13_API_Driven_KNN_Regressor_Distance_Metric_Tuning/notebook/Project_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import io
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
def fetch_api_dataset(url):
  try:
    response = requests.get(url)
    response.raise_for_status()
    data = pd.read_csv(io.StringIO(response.text))
    return data
  except requests.exceptions.RequestException as e:
    print("Error fetching data from API:", e)


In [ ]:
TRIP_TELEMETRY_API_ENDPOINT = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"

In [ ]:
df = fetch_api_dataset(TRIP_TELEMETRY_API_ENDPOINT)

In [ ]:
df.isna().sum()

,0
species,0
island,0
bill_length_mm,2
bill_depth_mm,2
flipper_length_mm,2
body_mass_g,2
sex,11


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB


In [ ]:
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [ ]:
df.dropna(inplace=True)

In [ ]:
df.rename(columns={
    'bill_length_mm': 'traffic_density_index',
    'bill_depth_mm': 'pickup_hour',
    'flipper_length_mm': 'distance_km',
    'body_mass_g': 'trip_duration_min'
},inplace=True)

In [ ]:
df.drop(columns=['species','island','sex'],inplace=True)

In [ ]:
df['trip_duration_min']=df['trip_duration_min']/100

In [ ]:
df.head()

,traffic_density_index,pickup_hour,distance_km,trip_duration_min
0,39.1,18.7,181.0,37.5
1,39.5,17.4,186.0,38.0
2,40.3,18.0,195.0,32.5
4,36.7,19.3,193.0,34.5
5,39.3,20.6,190.0,36.5


In [ ]:
max_val=df.max()
min_val = df.min()

In [ ]:
X = df.drop(['trip_duration_min'],axis=1)
y=df['trip_duration_min']

In [ ]:
y_bins = pd.qcut(y, q=5, labels=False)


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y_bins)

In [ ]:
X_train_size = X_train.shape[0]
X_test_size = X_test.shape[0]
y_train_mean = y_train.mean().round(2)
y_test_mean = y_test.mean().round(2)
print(f"Training set size: {X_train_size}")
print(f"Testing set size: {X_test_size}")
print(f"Mean of y_train: {y_train_mean}")
print(f"Mean of y_test: {y_test_mean}")

Training set size: 266
Testing set size: 67
Mean of y_train: 42.13
Mean of y_test: 41.82


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
knn = KNeighborsRegressor(n_neighbors=5,metric='minkowski',p=2)
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)
mae = round(mean_absolute_error(y_test, y_pred),2)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse).round(2)


2.71
3.55


In [92]:
k_values = [1, 3, 5, 7, 9, 15]
results=[]
for k in k_values:
  for p in [2,1]:
    model = KNeighborsRegressor(n_neighbors=k, metric='minkowski', p=p)
    model.fit(X_train_scaled, y_train)
    y_pred_new = model.predict(X_test_scaled)
    mae_new = round(mean_absolute_error(y_test, y_pred_new),2)
    mse_new = mean_squared_error(y_test, y_pred_new)
    rmse_new = np.sqrt(mse_new).round(2)
    results.append({
        'k':k,
        'p':p,
        'mae':mae_new,
        'rmse':rmse_new
    })
result_df=pd.DataFrame(results)

In [93]:
print(result_df)

     k  p   mae  rmse
0    1  2  3.15  4.19
1    1  1  3.24  4.32
2    3  2  2.94  3.74
3    3  1  2.91  3.67
4    5  2  2.71  3.55
5    5  1  2.73  3.60
6    7  2  2.67  3.43
7    7  1  2.66  3.51
8    9  2  2.59  3.41
9    9  1  2.58  3.37
10  15  2  2.48  3.24
11  15  1  2.42  3.17


In [95]:
best_mae = result_df.loc[result_df['mae'].idxmin()]
best_rmse = result_df.loc[result_df['rmse'].idxmin()]

print("Best by MAE:")
print(best_mae)

print("\nBest by RMSE:")
print(best_rmse)

Best by MAE:
k       15.00
p        1.00
mae      2.42
rmse     3.17
Name: 11, dtype: float64

Best by RMSE:
k       15.00
p        1.00
mae      2.42
rmse     3.17
Name: 11, dtype: float64


In [101]:
best_k = best_mae['k'].astype(int)
best_p = best_mae['p'].astype(int)

best_model = KNeighborsRegressor(n_neighbors=best_k, metric='minkowski', p=best_p)
best_model.fit(X_train_scaled, y_train)
y_pred_best = best_model.predict(X_test_scaled)

residuals = y_test - y_pred_best

abs_residuals = np.abs(residuals)



First 5 Predictions:
Sample 1 | Actual = 48.50 | Predicted = 52.50 | Absolute Residual = 4.00
Sample 2 | Actual = 39.50 | Predicted = 40.25 | Absolute Residual = 0.75
Sample 3 | Actual = 38.00 | Predicted = 36.53 | Absolute Residual = 1.47
Sample 4 | Actual = 50.00 | Predicted = 53.78 | Absolute Residual = 3.78
Sample 5 | Actual = 44.00 | Predicted = 46.47 | Absolute Residual = 2.47


In [103]:
print("\n========== API-DRIVEN KNN REGRESSOR & DISTANCE HYPERPARAMETER ENGINE ==========")

print("\nData Ingestion Status      : REST API Ingestion Successful (HTTP 200 OK)")
print(f"Master Dataset Records     : {df.shape[0]} Trip Records (Cleaned & Processed)")
print("Features Included          : 3 Continuous Distance & Traffic Metrics (distance_km, traffic_density_index, pickup_hour)")
print("Target Output              : trip_duration_min (Continuous Regression Target)")

print("\nModel Training Metrics:")
print(f"- Stratified Quantile Split: {X_train_size} Train Records / {X_test_size} Test Records")
print("- Feature Preprocessing    : StandardScaler Applied (Mean = 0.00, Std = 1.00)")
print(f"- Baseline Model (K=5, L2) : MAE = {mae:.2f} min | RMSE = {rmse:.2f} min")

print("\nHyperparameter Tuning Grid:")

best_metric = "Manhattan Distance (p=1)" if best_p == 1 else "Euclidean Distance (p=2)"

print(f"- Best Distance Metric     : {best_metric}")
print(f"- Optimal K-Neighbors      : K = {best_k}")
print(f"- Optimized Test Performance: MAE = {best_mae['mae']:.2f} min | RMSE = {best_mae['rmse']:.2f} min")

performance_gain = ((mae - best_mae['mae']) / mae) * 100
print(f"- Performance Gain         : {performance_gain:.2f}% Error Reduction over Baseline")

print("\nResidual Performance Summary:")
print(f"- Average Error Margin     : ±{best_mae['mae']:.2f} minutes per trip prediction")
print(f"- Maximum Absolute Residual: {abs_residuals.max():.2f} min")
print(f"- Minimum Absolute Residual: {abs_residuals.min():.2f} min")

print("\nConclusion:")
print("By streaming spatial telemetry over HTTP, feature scaling ensures distance metrics are not biased toward higher-magnitude features like distance_km. Hyperparameter tuning evaluates multiple K values and distance metrics, with the lowest observed test error obtained using the selected configuration.")


========== API-DRIVEN KNN REGRESSOR & DISTANCE HYPERPARAMETER ENGINE ==========

Data Ingestion Status      : REST API Ingestion Successful (HTTP 200 OK)
Master Dataset Records     : 333 Trip Records (Cleaned & Processed)
Features Included          : 3 Continuous Distance & Traffic Metrics (distance_km, traffic_density_index, pickup_hour)
Target Output              : trip_duration_min (Continuous Regression Target)

Model Training Metrics:
- Stratified Quantile Split: 266 Train Records / 67 Test Records
- Feature Preprocessing    : StandardScaler Applied (Mean = 0.00, Std = 1.00)
- Baseline Model (K=5, L2) : MAE = 2.71 min | RMSE = 3.55 min

Hyperparameter Tuning Grid:
- Best Distance Metric     : Manhattan Distance (p=1)
- Optimal K-Neighbors      : K = 15
- Optimized Test Performance: MAE = 2.42 min | RMSE = 3.17 min
- Performance Gain         : 10.70% Error Reduction over Baseline

Residual Performance Summary:
- Average Error Margin     : ±2.42 minutes per trip prediction
- Maximu